# CerviScan IA - Pipeline de Diagnostic Assiste

Ce notebook constitue le Pipeline Maitre du projet CerviScan IA. Il suit rigoureusement les Regles d'Or du ML Professionnel definies dans le projet.

### Objectifs :
1. Exploration (EDA) : Comprendre nos donnees (Regles 2, 3, 13).
2. Preparation : Organiser les donnees pour l'entrainement (Regle 4).
3. Entrainement : Developper une baseline solide avec EfficientNet (Regles 9, 15).
4. Evaluation : Mesurer la performance reelle (Regles 10, 11).

## Configuration Initiale
On prepare les chemins et les bibliotheques.

In [ ]:
import os
import sys
from pathlib import Path
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

# On definit la racine du projet et l'environnement virtuel dynamiquement
REPO_ROOT = Path(os.getcwd()).parent.parent.resolve()
VENV_SITE = Path(os.getcwd()).parent / ".venv" / "Lib" / "site-packages"

if str(REPO_ROOT) not in sys.path: sys.path.insert(0, str(REPO_ROOT))
if VENV_SITE.exists() and str(VENV_SITE) not in sys.path: sys.path.insert(0, str(VENV_SITE))

from ml.src.utils.config import load_and_merge
from ml.src.training.trainer import train, evaluate
from ml.src.utils.seed import set_seed

# On fixe la graine aleatoire pour la reproductibilite (Regle 4)
set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Racine du projet : {REPO_ROOT}")
print(f"Dispositif utilise : {DEVICE}")

## Etape 1 : Exploration des Donnees (EDA)
Regle 2 : Connais tes donnees comme ta poche.
Regle 13 : Identifie le desequilibre des classes.

In [ ]:
DATA_DIR = REPO_ROOT / 'ml' / 'data'
counts_path = REPO_ROOT / 'ml' / 'reports' / 'metrics' / 'class_counts.csv'

if counts_path.exists():
    df_counts = pd.read_csv(counts_path)
    class_map = {0: 'Type_1', 1: 'Type_2', 2: 'Type_3'}
    df_counts['class_name'] = df_counts['label'].map(class_map)
    
    # Visualisation de la distribution
    plt.figure(figsize=(8, 5))
    sns.barplot(data=df_counts, x='class_name', y='count', hue='class_name', palette='magma', legend=False)
    plt.title("Distribution des classes (CerviScan IA)")
    plt.show()
    
    total = df_counts['count'].sum()
    for _, row in df_counts.iterrows():
        print(f"Modele {row['class_name']} : {row['count']} images ({row['count']/total:.1%})")
else:
    print("Attention : fichier class_counts.csv manquant. Executez prepare_dataset.py.")

## Etape 2 : Configuration du Pipeline
On fusionne les fichiers YAML pour creer la configuration d'entrainement.

In [ ]:
config_files = [
    REPO_ROOT / 'ml' / 'configs' / 'base.yaml',
    REPO_ROOT / 'ml' / 'configs' / 'data.yaml',
    REPO_ROOT / 'ml' / 'configs' / 'train.yaml',
    REPO_ROOT / 'ml' / 'configs' / 'eval.yaml',
]

cfg = load_and_merge(config_files)

# On limite a 5 epoques pour un test rapide du pipeline
cfg['training']['epochs'] = 5 
print("Configuration master chargee avec succes.")

## Etape 3 : Entrainement de la Baseline
Regle 9 : Commence par une baseline simple (EfficientNet-B4).

In [ ]:
print("Demarrage de l'entrainement...")
results = train(cfg)

print(f"\nMeilleure epoque : {results.best_epoch}")
print(f"Meilleur F1-Score : {results.best_metric:.4f}")

## Etape 4 : Analyse et Evaluation
Regle 11 : Valide ta performance sur des donnees jamais vues (Test Set).

In [ ]:
# 1. Trace des courbes
history = pd.DataFrame(results.history)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train')
plt.plot(history['val_loss'], label='Val')
plt.title("Evolution de la Perte (Loss)")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_f1_macro'], label='Train')
plt.plot(history['val_f1_macro'], label='Val')
plt.title("Macro F1-Score")
plt.legend()
plt.show()

# 2. Evaluation finale
best_pt = REPO_ROOT / 'ml' / 'models' / 'checkpoints' / 'best.pt'
if best_pt.exists():
    test_metrics = evaluate(cfg, best_pt, split='test')
    print("\nPERFORMANCE FINALE SUR LE TEST SET :")
    for k, v in test_metrics.items():
        print(f"   - {k} : {v:.4f}")

### Conclusion
La baseline est operationnelle. Les prochaines etapes consisteront a :
- Optimiser les hypermetres.
- Exporter le modele pour la PWA.
- Integrer les retours cliniques.